# Lab 12: Interactive GenAI Playground

## Overview
In this lab you will build a Streamlit web application that integrates Bedrock models, Knowledge Bases, Guardrails, and prompt engineering techniques into an interactive playground for testing and learning. The app provides a multi-tab interface covering streaming chat, RAG with Knowledge Bases, side-by-side model comparison, real-time guardrail testing, and prompt engineering experimentation.

## Learning Objectives
- Build a Streamlit app with Amazon Bedrock integration
- Implement streaming chat with the ConverseStream API
- Connect to Knowledge Bases for interactive RAG queries
- Test Guardrails in real-time with pre-loaded and custom prompts
- Compare multiple foundation models side-by-side

## Exam Domain
**Building Generative AI Applications (30%)** — This lab covers building end-to-end generative AI applications with a user interface, integrating multiple Bedrock features (Converse, ConverseStream, Knowledge Bases, Guardrails) into a cohesive application that demonstrates production patterns for model selection, streaming, RAG, and content safety.

## Architecture Diagram
![Lab 12 Architecture](../../assets/diagrams/lab-12-interactive-playground.png)

In [ ]:
%pip install boto3 streamlit --quiet

In [ ]:
import boto3, json, os, time

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock_runtime = session.client("bedrock-runtime")
bedrock = session.client("bedrock")
bedrock_agent = session.client("bedrock-agent")
bedrock_agent_runtime = session.client("bedrock-agent-runtime")

# Model registry
MODELS = {
    "Claude Sonnet 4.5": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "Claude Haiku 4.5": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
    "Llama 3 8B": "meta.llama3-8b-instruct-v1:0",
    "Mistral 7B": "mistral.mistral-7b-instruct-v0:2",
}

print(f"Region: {REGION}")
print(f"Models: {list(MODELS.keys())}")

## A. Chat Tab — Streaming with ConverseStream

The Chat tab implements a conversational interface using Streamlit's `st.chat_message` and `st.chat_input` components. The key integration is with Bedrock's `converse_stream` API, which returns a streaming response that we feed into `st.write_stream` for real-time token rendering.

**Key patterns:**
- `st.session_state` stores the chat history across reruns (Streamlit re-executes the entire script on each interaction)
- `converse_stream` returns an event stream — we yield text deltas from `contentBlockDelta` events
- The `metadata` event at the end of the stream contains token usage statistics
- A sidebar provides model selection, temperature, max tokens, and an optional system prompt

In [ ]:
# A1. Test the ConverseStream API directly (non-Streamlit version)
# This shows the raw streaming pattern that the Chat tab wraps

model_id = MODELS["Claude Sonnet 4.5"]
messages = [{"role": "user", "content": [{"text": "What is Amazon Bedrock in one sentence?"}]}]

response = bedrock_runtime.converse_stream(
    modelId=model_id,
    messages=messages,
    inferenceConfig={"maxTokens": 200, "temperature": 0.7},
)

# Process the stream
full_text = ""
for event in response["stream"]:
    if "contentBlockDelta" in event:
        chunk = event["contentBlockDelta"]["delta"].get("text", "")
        full_text += chunk
        print(chunk, end="", flush=True)
    if "metadata" in event:
        usage = event["metadata"].get("usage", {})

print(f"\n\nTokens — input: {usage.get('inputTokens')}, output: {usage.get('outputTokens')}")

In [ ]:
# A2. The Streamlit Chat tab code
# In app.py, the streaming generator wraps converse_stream for st.write_stream:

chat_tab_code = '''
def stream_converse(model_id, messages, system=None, max_tokens=512, temperature=0.7):
    kwargs = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
    }
    if system:
        kwargs["system"] = [{"text": system}]
    response = bedrock_runtime.converse_stream(**kwargs)
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"]["delta"]
            if "text" in delta:
                yield delta["text"]
        if "metadata" in event:
            st.session_state["_last_usage"] = event["metadata"].get("usage", {})

# Usage in the Chat tab:
# with st.chat_message("assistant"):
#     response_text = st.write_stream(
#         stream_converse(model_id, converse_messages, system=sys,
#                         max_tokens=max_tokens, temperature=temperature)
#     )
'''
print(chat_tab_code)

## B. RAG Tab — Knowledge Base Integration

The RAG tab connects to an existing Bedrock Knowledge Base (created in Lab 05) and uses the `retrieve_and_generate` API to answer questions with retrieved context. This API handles both retrieval and generation in a single call.

**Key patterns:**
- `list_knowledge_bases` discovers available KBs — the app gracefully handles the case when none exist
- `retrieve_and_generate` takes a query, retrieves relevant chunks from the KB's vector store, and generates a response using the specified model
- The response includes `citations` with `retrievedReferences` — each reference contains the chunk text and its source location
- The model ARN format for Bedrock is `arn:aws:bedrock:{region}::foundation-model/{model_id}`

In [ ]:
# B1. List available Knowledge Bases

try:
    kb_response = bedrock_agent.list_knowledge_bases(maxResults=10)
    kb_list = kb_response.get("knowledgeBaseSummaries", [])
    if kb_list:
        for kb in kb_list:
            print(f"KB: {kb['name']} (ID: {kb['knowledgeBaseId']}, Status: {kb['status']})")
    else:
        print("No Knowledge Bases found. Run Lab 05 first to create one.")
except Exception as e:
    print(f"Error listing Knowledge Bases: {e}")

In [ ]:
# B2. Test RetrieveAndGenerate (only works if a KB exists from Lab 05)

if kb_list:
    KB_ID = kb_list[0]["knowledgeBaseId"]
    model_arn = f"arn:aws:bedrock:{REGION}::foundation-model/{MODELS['Claude Sonnet 4.5']}"

    rag_response = bedrock_agent_runtime.retrieve_and_generate(
        input={"text": "What is Amazon Bedrock?"},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": KB_ID,
                "modelArn": model_arn,
            },
        },
    )

    print("Answer:", rag_response["output"]["text"])
    print(f"\nCitations: {len(rag_response.get('citations', []))}")

    for i, citation in enumerate(rag_response.get("citations", [])):
        for j, ref in enumerate(citation.get("retrievedReferences", [])):
            print(f"\n--- Chunk {i+1}.{j+1} ---")
            print(ref.get("content", {}).get("text", "N/A")[:300])
else:
    print("Skipping — no Knowledge Base available. Run Lab 05 first.")

## C. Model Compare Tab — Side-by-Side Evaluation

The Model Compare tab sends the same prompt to multiple models simultaneously and displays results side by side. This is a practical way to evaluate model quality, latency, and token efficiency for your specific use case.

**Key patterns:**
- The Converse API provides a unified interface across all Bedrock models — the same message format works for Claude, Llama, and Mistral
- `st.columns` creates a responsive side-by-side layout
- Timing each call with `time.time()` provides latency comparison
- Token counts from the `usage` field in the response let you compare cost efficiency

In [ ]:
# C1. Compare models programmatically

compare_prompt = "Explain what a vector database is in exactly two sentences."
compare_models = ["Claude Sonnet 4.5", "Llama 3 8B"]

messages = [{"role": "user", "content": [{"text": compare_prompt}]}]

print(f"Prompt: {compare_prompt}\n")
print("=" * 70)

for model_name in compare_models:
    model_id = MODELS[model_name]
    start = time.time()

    try:
        resp = bedrock_runtime.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={"maxTokens": 512, "temperature": 0.7},
        )
        elapsed = time.time() - start
        output_text = resp["output"]["message"]["content"][0]["text"]
        usage = resp.get("usage", {})

        print(f"\n{model_name}")
        print(f"-" * len(model_name))
        print(output_text)
        print(f"Time: {elapsed:.2f}s | Input: {usage.get('inputTokens')} | Output: {usage.get('outputTokens')}")
        print("=" * 70)

    except Exception as e:
        print(f"\n{model_name}: Error — {e}")
        print("=" * 70)

## D. Guardrails Tab — Real-Time Content Safety Testing

The Guardrails tab lets you test a Bedrock Guardrail interactively with pre-loaded test prompts that exercise different filter types. The app discovers an existing guardrail or creates a default one with denied topics, PII filters, and content filters.

**Key patterns:**
- `list_guardrails` searches for an existing guardrail; `create_guardrail` builds one if none exists
- `guardrailConfig` is passed to the Converse API with `trace: "enabled"` for full evaluation details
- The `stopReason` field distinguishes `guardrail_intervened` (blocked) from `end_turn` (allowed)
- The trace object contains detailed input/output assessments for every policy type

In [ ]:
# D1. Find or create a guardrail for testing

guardrail_id = None
guardrail_version = "DRAFT"

# Check for existing guardrail
try:
    existing = bedrock.list_guardrails(maxResults=10)
    for g in existing.get("guardrails", []):
        if g["name"] == "PlaygroundGuardrail":
            guardrail_id = g["id"]
            print(f"Found existing guardrail: {guardrail_id}")
            break
except Exception as e:
    print(f"Error listing guardrails: {e}")

# Create if not found
if not guardrail_id:
    resp = bedrock.create_guardrail(
        name="PlaygroundGuardrail",
        description="Default guardrail for the GenAI Playground",
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "InvestmentAdvice",
                    "definition": "Providing specific investment, stock trading, or financial planning recommendations",
                    "examples": [
                        "Should I buy AMZN stock?",
                        "What stocks should I invest in?",
                        "Is now a good time to invest in tech?",
                    ],
                    "type": "DENY",
                }
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "US_SOCIAL_SECURITY_NUMBER", "action": "BLOCK"},
            ]
        },
        contentPolicyConfig={
            "filtersConfig": [
                {"type": "SEXUAL", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "HATE", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "INSULTS", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "MISCONDUCT", "inputStrength": "HIGH", "outputStrength": "HIGH"},
                {"type": "PROMPT_ATTACK", "inputStrength": "HIGH", "outputStrength": "NONE"},
            ]
        },
        blockedInputMessaging="Your request was blocked by the guardrail.",
        blockedOutputsMessaging="The model response was blocked by the guardrail.",
    )
    guardrail_id = resp["guardrailId"]
    print(f"Created guardrail: {guardrail_id}")

print(f"Guardrail ID: {guardrail_id}, Version: {guardrail_version}")

In [ ]:
# D2. Test with safe, denied-topic, and PII prompts

test_cases = [
    ("Safe", "What is Amazon Bedrock?"),
    ("Denied Topic", "Should I invest in AMZN stock?"),
    ("PII", "My email is john@example.com, my SSN is 123-45-6789. Can you help me?"),
]

for label, prompt_text in test_cases:
    print(f"\n{'=' * 60}")
    print(f"Test: {label}")
    print(f"Prompt: {prompt_text}")
    print("-" * 60)

    messages = [{"role": "user", "content": [{"text": prompt_text}]}]

    try:
        resp = bedrock_runtime.converse(
            modelId=MODELS["Claude Sonnet 4.5"],
            messages=messages,
            inferenceConfig={"maxTokens": 512, "temperature": 0.7},
            guardrailConfig={
                "guardrailIdentifier": guardrail_id,
                "guardrailVersion": guardrail_version,
                "trace": "enabled",
            },
        )

        stop_reason = resp.get("stopReason", "unknown")
        output_text = resp["output"]["message"]["content"][0]["text"]

        if stop_reason == "guardrail_intervened":
            print(f"Action: BLOCKED (stop reason: {stop_reason})")
        else:
            print(f"Action: ALLOWED (stop reason: {stop_reason})")

        print(f"Output: {output_text[:200]}")

        # Show trace summary
        trace = resp.get("trace", {}).get("guardrail", {})
        if trace:
            print(f"Trace keys: {list(trace.keys())}")

    except Exception as e:
        print(f"Error: {e}")

## E. Prompt Lab Tab — Prompt Engineering Techniques

The Prompt Lab tab provides pre-built templates for three common prompt engineering techniques: zero-shot, few-shot, and chain-of-thought. Each template has a variable placeholder that the user fills in, making it easy to experiment with how different prompting strategies affect model output.

**Key patterns:**
- **Zero-shot** gives no examples — the model relies entirely on its pre-training to understand the task
- **Few-shot** provides 2-3 examples before the actual input, teaching the model the expected format
- **Chain-of-thought** instructs the model to reason step by step, improving accuracy on complex tasks
- Showing the full prompt in an expander helps users understand exactly what the model receives

In [ ]:
# E1. Zero-shot classification

review = "This film was absolutely brilliant and kept me on the edge of my seat."
zero_shot_prompt = f'Classify the following movie review as POSITIVE or NEGATIVE.\n\nReview: "{review}"\n\nClassification:'

print("=== Zero-Shot Prompt ===")
print(zero_shot_prompt)
print()

resp = bedrock_runtime.converse(
    modelId=MODELS["Claude Sonnet 4.5"],
    messages=[{"role": "user", "content": [{"text": zero_shot_prompt}]}],
    inferenceConfig={"maxTokens": 100, "temperature": 0.7},
)
print("Response:", resp["output"]["message"]["content"][0]["text"])
print(f"Tokens: {resp.get('usage', {})}")

In [ ]:
# E2. Few-shot classification

review = "The movie was dull and I almost fell asleep."
few_shot_prompt = (
    'Classify each movie review as POSITIVE or NEGATIVE.\n\n'
    'Review: "I loved every minute of this movie!"\nClassification: POSITIVE\n\n'
    'Review: "Terrible acting and a boring plot."\nClassification: NEGATIVE\n\n'
    'Review: "A masterpiece of modern cinema."\nClassification: POSITIVE\n\n'
    f'Review: "{review}"\nClassification:'
)

print("=== Few-Shot Prompt ===")
print(few_shot_prompt)
print()

resp = bedrock_runtime.converse(
    modelId=MODELS["Claude Sonnet 4.5"],
    messages=[{"role": "user", "content": [{"text": few_shot_prompt}]}],
    inferenceConfig={"maxTokens": 100, "temperature": 0.7},
)
print("Response:", resp["output"]["message"]["content"][0]["text"])
print(f"Tokens: {resp.get('usage', {})}")

In [ ]:
# E3. Chain-of-thought reasoning

problem = "A store sells apples for $1.50 each. If you buy 5 or more, you get a 20% discount. How much do 7 apples cost?"
cot_prompt = (
    "Solve the following problem step by step. Show your reasoning before giving the final answer.\n\n"
    f"Problem: {problem}\n\n"
    "Step-by-step solution:"
)

print("=== Chain-of-Thought Prompt ===")
print(cot_prompt)
print()

resp = bedrock_runtime.converse(
    modelId=MODELS["Claude Sonnet 4.5"],
    messages=[{"role": "user", "content": [{"text": cot_prompt}]}],
    inferenceConfig={"maxTokens": 512, "temperature": 0.7},
)
print("Response:", resp["output"]["message"]["content"][0]["text"])
print(f"Tokens: {resp.get('usage', {})}")

## F. Run the App

The complete Streamlit application is in `app.py` in this directory. To run it:

```bash
cd labs/12-interactive-playground
streamlit run app.py
```

This will open a browser tab at `http://localhost:8501` with the full playground.

**What to expect:**
- **Chat tab** — Select a model from the sidebar, type a message, and see streaming responses in real time. Try switching models and adjusting temperature to see how output changes.
- **RAG tab** — If you completed Lab 05, select your Knowledge Base and ask questions. The app shows the generated answer and lets you expand the retrieved chunks.
- **Model Compare tab** — Enter a prompt, select 2-4 models, and compare their responses side by side with latency and token counts.
- **Guardrails tab** — Use the pre-loaded test prompts to see how denied topics, PII filters, and content filters work. Try custom inputs to explore edge cases.
- **Prompt Lab tab** — Switch between zero-shot, few-shot, and chain-of-thought templates. Modify the input and observe how each technique affects the output.

## Cleanup

In [ ]:
# Delete the playground guardrail (if created in this notebook)
if guardrail_id:
    try:
        bedrock.delete_guardrail(guardrailIdentifier=guardrail_id)
        print(f"Deleted guardrail: {guardrail_id}")
    except Exception as e:
        print(f"Error deleting guardrail: {e}")

print("\nCleanup complete.")

## Key Takeaways

1. **Streamlit provides a rapid way to build interactive GenAI prototypes with Python** — its reactive model (re-run-on-interaction) paired with `st.session_state` for persistence maps naturally to chat and form-based AI applications, letting you go from notebook code to a shareable web app in hours
2. **The ConverseStream API enables real-time token streaming that dramatically improves perceived latency** — instead of waiting for the full response, users see tokens appear as they are generated, and `st.write_stream` handles the rendering automatically from a Python generator
3. **The Converse API unifies model interaction across providers** — the same message format works for Claude, Llama, and Mistral models, making it straightforward to build model-agnostic applications and switch models without changing application code
4. **RetrieveAndGenerate provides a single API call for end-to-end RAG** — it handles retrieval from the Knowledge Base vector store and generation with the specified model, returning both the answer and citations with source references for transparency
5. **Guardrails can be applied to any Converse API call by adding a guardrailConfig parameter** — the guardrail evaluates both input and output against all configured policies (content filters, denied topics, PII detection) and the trace provides detailed information about what triggered each filter

## Key Concepts

| Concept | Definition |
|---------|------------|
| Streamlit | An open-source Python framework for building interactive web applications with simple Python scripts — each user interaction triggers a full script re-run, with `st.session_state` providing persistence across reruns |
| st.chat_message | A Streamlit component that renders chat bubbles with role-based styling (user vs assistant), used with `st.chat_input` to build conversational interfaces |
| st.write_stream | A Streamlit function that accepts a generator and renders yielded text chunks in real time, enabling streaming output from language models |
| ConverseStream | The streaming variant of the Bedrock Converse API that returns an event stream with `contentBlockDelta` events containing text chunks and a final `metadata` event with token usage |
| RetrieveAndGenerate | A Bedrock Knowledge Base API that combines document retrieval from a vector store with model-based generation in a single call, returning both the answer and citations with source references |
| GuardrailConfig | A parameter in the Converse API that applies a Bedrock Guardrail to the request, specifying the guardrail ID, version, and optional trace enablement for detailed policy evaluation reporting |
| Model comparison | A pattern for evaluating multiple foundation models by sending the same prompt to each and comparing response quality, latency, and token usage side by side |

## Exam Preparation — Common Exam Question Patterns

**Q: A team wants to build a prototype GenAI application that lets non-technical stakeholders test different models and prompting strategies. What approach minimizes development time while providing interactive model access?**
A: Build a **Streamlit application** that uses the **Converse API** for model interaction. Streamlit lets developers create interactive web UIs with pure Python, and the Converse API provides a unified interface across all Bedrock models. The team can add model selection dropdowns, parameter sliders, and side-by-side comparison views without frontend development. For streaming responses, use `converse_stream` with `st.write_stream`.

**Q: How does ConverseStream differ from Converse, and when should you use each?**
A: **Converse** returns the complete response in a single API call — use it when you need the full response before processing (e.g., batch processing, model comparison, or when applying post-processing). **ConverseStream** returns an event stream that delivers text chunks as they are generated — use it for interactive applications where perceived latency matters, such as chat interfaces. ConverseStream events include `contentBlockDelta` (text chunks) and `metadata` (token usage at the end).

**Q: An application uses RetrieveAndGenerate with a Knowledge Base. How can the application verify that the model's answer is grounded in the retrieved documents?**
A: The `retrieve_and_generate` response includes a `citations` field containing `retrievedReferences` for each citation. Each reference has the `content.text` (the retrieved chunk) and `location` (the source document). The application should display these citations to users and can programmatically verify that key claims in the answer appear in the retrieved chunks. For automated grounding checks, add a **contextual grounding** guardrail that evaluates response-to-source alignment.

**Q: What is the benefit of using the Converse API instead of model-specific InvokeModel calls when building a multi-model application?**
A: The **Converse API provides a unified message format** that works across all Bedrock models (Claude, Llama, Mistral, Titan, etc.) without model-specific request/response formatting. This means the same application code can call any model by changing only the `modelId` parameter. It also provides consistent `usage` metadata (input/output tokens) and standardized `stopReason` values. For model comparison applications, this eliminates the need to maintain separate serialization logic for each model provider.

**Q: How should guardrails be applied in a production application that uses the Converse API?**
A: Pass a `guardrailConfig` parameter to the Converse API call with the `guardrailIdentifier` (guardrail ID) and `guardrailVersion`. The guardrail evaluates both the input and the model's output against all configured policies. Check the `stopReason` field — `guardrail_intervened` indicates the request or response was blocked. Enable `trace` during development to see detailed evaluation results for each policy type. In production, log the trace data to CloudWatch for monitoring false positives and tuning filter thresholds.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Claude Sonnet 4.5 — Section A (ConverseStream test) | 1 call, ~300 tokens | ~$0.01 |
| Claude Sonnet 4.5 — Section C (model compare, 2 models) | 2 calls, ~1K tokens | ~$0.02 |
| Claude Sonnet 4.5 — Section D (guardrail tests, 3 prompts) | 3 calls, ~1.5K tokens | ~$0.03 |
| Claude Sonnet 4.5 — Section E (3 prompt techniques) | 3 calls, ~1.5K tokens | ~$0.03 |
| Llama 3 8B — Section C (model compare) | 1 call, ~500 tokens | ~$0.01 |
| Bedrock Guardrails — evaluation charges | 3 evaluations | ~$0.25 |
| Streamlit app usage (interactive) | Varies by testing | ~$0.15 |
| **Total** | | **~$0.50** |

Costs depend on how much you use the Streamlit app interactively. The notebook cells above have minimal cost. The Streamlit app costs scale with the number of messages sent and models tested. Guardrail evaluation charges apply per assessment.